In [1]:
import calendar
from datetime import timedelta, datetime
import pprint

import ee
from IPython.display import Image, display
import ipyplot
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import openet.core

ee.Initialize()

In [2]:
v21_model_collections = {
    'DisALEXI': ee.ImageCollection('projects/openet/assets/disalexi/california/cimis/monthly/v2_1'),
    'EEMETRIC': ee.ImageCollection('projects/openet/assets/eemetric/california/cimis/monthly/v2_1'),
    'GEESEBAL': ee.ImageCollection('projects/openet/assets/geesebal/california/cimis/monthly/v2_1'),
    'PTJPL': ee.ImageCollection('projects/openet/assets/ptjpl/california/cimis/monthly/v2_1'),
    'SIMS': ee.ImageCollection('projects/openet/assets/sims/california/cimis/monthly/v2_1'),
    'SSEBop': ee.ImageCollection('projects/openet/assets/ssebop/california/cimis/monthly/v2_1'),
    'Ensemble': ee.ImageCollection('projects/openet/assets/ensemble/california/cimis/monthly/v2_1'),
}
v20_model_collections = {
    'DisALEXI': ee.ImageCollection('projects/openet/assets/disalexi/california/cimis/monthly/v2_0'),
    'EEMETRIC': ee.ImageCollection('projects/openet/assets/eemetric/california/cimis/monthly/v2_0'),
    'GEESEBAL': ee.ImageCollection('projects/openet/assets/geesebal/california/cimis/monthly/v2_0'),
    'PTJPL': ee.ImageCollection('projects/openet/assets/ptjpl/california/cimis/monthly/v2_0'),
    'SIMS': ee.ImageCollection('projects/openet/assets/sims/california/cimis/monthly/v2_0'),
    'SSEBop': ee.ImageCollection('projects/openet/assets/ssebop/california/cimis/monthly/v2_0'),
    'Ensemble': ee.ImageCollection('projects/openet/assets/ensemble/california/cimis/monthly/v2_0'),
}


models = ['DisALEXI', 'EEMETRIC', 'GEESEBAL', 'PTJPL', 'SIMS', 'SSEBop']

model_band_name = 'et'
ensemble_band_name = 'et_ensemble_mad'

years = [2024]
# years = list(reversed(range(2004, 2025)))
#years = list(reversed(range(2016, 2025)))
#years = list(reversed(range(2020, 2025)))
#years = list(range(2016, 2026))

region = ee.Geometry.BBox(-124.5, 32.4, -114.0, 42)
image_size = 400
thumb_args = {'region': region, 'dimensions': image_size}

et_palette = ['DEC29B', 'E6CDA1', 'EDD9A6', 'F5E4A9', 'FFF4AD', 'C3E683', '6BCC5C', '3BB369', '20998F', '1C8691', '16678A', '114982', '0B2C7A']
viridis = ['440154', '433982', '30678D', '218F8B', '36B677', '8ED542', 'FDE725']
rb_palette = ['red', 'white', 'blue']
ryb_palette = ['red', 'lightyellow', 'blue']

land_mask = ee.Image('projects/openet/assets/features/water_mask').Not()
# Apply the NLCD/NALCMS water mask (anywhere it is water, set the ocean mask 
land_mask = land_mask.where(ee.Image("USGS/NLCD_RELEASES/2020_REL/NALCMS").unmask(18).eq(18), 0)
# land_mask = land_mask.And(ee.Image("USGS/NLCD_RELEASES/2020_REL/NALCMS").unmask(18).neq(18))
# # land_mask = ee.Image('projects/openet/assets/meteorology/conus404/ancillary/land_mask')

def mask_count_zero(img):
    mask_img = img.select(['count']).gt(0)
    return img.updateMask(mask_img)


### Ensemble Annual Sums 

In [3]:
model_name = 'Ensemble'
urls = []
labels = []
for year in years:
    v20_img = v20_model_collections[model_name].filterDate(f'{year}-01-01', f'{year+1}-01-01').select([ensemble_band_name]).sum()
    v21_img = v21_model_collections[model_name].filterDate(f'{year}-01-01', f'{year+1}-01-01').select([ensemble_band_name]).sum()
    diff_url = (
        v21_img.subtract(v20_img)
        .visualize(min=-200, max=200, palette=rb_palette).getThumbURL(thumb_args)
    )
    urls.append(diff_url)
    labels.append(f'{model_name} - {year} - Change in Total Annual ET')
    
ipyplot.plot_images(urls, labels=labels, show_url=False, img_width=image_size)


### Ensemble Monthly Count

In [4]:
model_name = 'Ensemble'
urls = []
labels = []
for year in years:
    v20_img = v20_model_collections[model_name].filterDate(f'{year}-01-01', f'{year+1}-01-01').select([ensemble_band_name]).count()
    v21_img = v21_model_collections[model_name].filterDate(f'{year}-01-01', f'{year+1}-01-01').select([ensemble_band_name]).count()
    diff_url = (
        v21_img.subtract(v20_img)
        .visualize(min=-3, max=3, palette=ryb_palette).getThumbURL(thumb_args)
    )
    urls.append(diff_url)
    labels.append(f'{model_name} - {year} - Change in Monthly Image Counts')
    
ipyplot.plot_images(urls, labels=labels, show_url=False, img_width=image_size)

### Ensemble Average Model Count

In [5]:
model_name = 'Ensemble'
urls = []
labels = []
for year in years:
    v20_img = v20_model_collections[model_name].filterDate(f'{year}-01-01', f'{year+1}-01-01').select(['et_ensemble_mad_count']).mean()
    v21_img = v21_model_collections[model_name].filterDate(f'{year}-01-01', f'{year+1}-01-01').select(['et_ensemble_mad_count']).mean()
    diff_url = (
        v21_img.subtract(v20_img)
        .visualize(min=-1, max=1, palette=rb_palette).getThumbURL(thumb_args)
    )
    urls.append(diff_url)
    labels.append(f'{model_name} - {year} - Change in Annual Scene Counts')
    
ipyplot.plot_images(urls, labels=labels, show_url=False, img_width=image_size)

### Ensemble Monthly Mask

In [6]:
# CGM - This one should show changes in the masking but doesn't work that well
# model_name = 'Ensemble'
# year = 2024
# urls = []
# labels = []
# for month in range(1, 13):
#     month_date = ee.Date(f'{year}-{month:02d}-01')
#     v20_img = v20_model_collections[model_name].filterDate(month_date, month_date.advance(1, 'month')).select([ensemble_band_name]).count()
#     v21_img = v21_model_collections[model_name].filterDate(month_date, month_date.advance(1, 'month')).select([ensemble_band_name]).count()
#     diff_url = (
#         v21_img.mask().subtract(v20_img.mask())
#         .visualize(min=-1, max=1, palette=rb_palette).getThumbURL(thumb_args)
#     )
#     urls.append(diff_url)
#     labels.append(f'{model_name} - {year} {calendar.month_abbr[month]} - Change in Monthly Data Mask')
    
# ipyplot.plot_images(urls, labels=labels, show_url=False, img_width=image_size)

### Model Annual Sums

In [7]:
for year in years:
    print(year)
    urls = []
    labels = []
    for model_name in models:
        v20_img = v20_model_collections[model_name].filterDate(f'{year}-01-01', f'{year+1}-01-01').select([model_band_name]).sum()
        v21_img = v21_model_collections[model_name].filterDate(f'{year}-01-01', f'{year+1}-01-01').select([model_band_name]).sum()
        diff_url = (
            v21_img.subtract(v20_img)
            .visualize(min=-400, max=400, palette=rb_palette).getThumbURL(thumb_args)
        )
        urls.append(diff_url)
        labels.append(f'{model_name} - {year} - Change in Total Annual ET')

    ipyplot.plot_images(urls, labels=labels, show_url=False, img_width=image_size)
    # break
    

2024


### Monthly Count

In [8]:
for year in years:
    print(year)
    urls = []
    labels = []
    for model_name in models:
        v20_img = v20_model_collections[model_name].filterDate(f'{year}-01-01', f'{year+1}-01-01').select([model_band_name]).count()
        v21_img = v21_model_collections[model_name].filterDate(f'{year}-01-01', f'{year+1}-01-01').select([model_band_name]).count()
        diff_url = (
            v21_img.subtract(v20_img)
            .visualize(min=-4, max=4, palette=ryb_palette).getThumbURL(thumb_args)
        )
        urls.append(diff_url)
        labels.append(f'{model_name} - {year} - Change in Monthly Image Counts')
        
    ipyplot.plot_images(urls, labels=labels, show_url=False, img_width=image_size)
    # break
    

2024


### Max monthly

In [9]:
for year in years:
    print(year)
    urls = []
    labels = []
    for model_name in models:
        v20_img = v20_model_collections[model_name].filterDate(f'{year}-01-01', f'{year+1}-01-01').select([model_band_name]).max()
        v21_img = v21_model_collections[model_name].filterDate(f'{year}-01-01', f'{year+1}-01-01').select([model_band_name]).max()
        diff_url = (
            v21_img.subtract(v20_img)
            .visualize(min=-100, max=100, palette=rb_palette).getThumbURL(thumb_args)
        )
        urls.append(diff_url)
        labels.append(f'{model_name} - {year} - Change in Max Monthly ET')

    ipyplot.plot_images(urls, labels=labels, show_url=False, img_width=image_size)
    break


2024


### Min monthly

In [10]:
for year in years:
    print(year)
    urls = []
    labels = []
    for model_name in models:
        v20_img = v20_model_collections[model_name].filterDate(f'{year}-01-01', f'{year+1}-01-01').select([model_band_name]).min()
        v21_img = v21_model_collections[model_name].filterDate(f'{year}-01-01', f'{year+1}-01-01').select([model_band_name]).min()
        diff_url = (
            v21_img.subtract(v20_img)
            .visualize(min=-100, max=100, palette=rb_palette).getThumbURL(thumb_args)
        )
        urls.append(diff_url)
        labels.append(f'{model_name} - {year} - Change in Min Monthly ET')

    ipyplot.plot_images(urls, labels=labels, show_url=False, img_width=image_size)
    break


2024


### Model Monthly Sums

In [11]:
for year in years:
    print(year)
    for month in range(1, 13):
        month_date = ee.Date(f'{year}-{month:02d}-01')
        urls = []
        labels = []
        for model_name in models:
            v20_img = v20_model_collections[model_name].filterDate(month_date, month_date.advance(1, 'month')).select([model_band_name]).sum()
            v21_img = v21_model_collections[model_name].filterDate(month_date, month_date.advance(1, 'month')).select([model_band_name]).sum()
            diff_url = (
                v21_img.subtract(v20_img)
                .visualize(min=-50, max=50, palette=rb_palette).getThumbURL(thumb_args)
            )
            urls.append(diff_url)
            labels.append(f'{model_name} - {year} {calendar.month_abbr[month]} - Change in Monthly ET')
        ipyplot.plot_images(urls, labels=labels, show_url=False, img_width=image_size)
        
    # break


2024
